In [1]:
from pathlib import Path
import copy

TRACKMAN_EXCEL = Path(r"C:\Users\brend\OneDrive\School OneDrive\2025 CCBL Trackman Data.xlsx")
OUTPUT_DIR = Path("Falmouth_Pitcher_Reports")
PITCHERS = ["Reich, Ryan", "Harrington, Cooper", "Shurtleff, Thomas", "Arther, Adam", "Doran, Scott"]

if not TRACKMAN_EXCEL.is_file():
    TRACKMAN_EXCEL = next((p for p in [Path("2025 CCBL Trackman Data(1).xlsx"), Path("/mnt/data/2025 CCBL Trackman Data(1).xlsx")] if p.is_file()), TRACKMAN_EXCEL)

if not TRACKMAN_EXCEL.is_file():
    raise FileNotFoundError(f"TrackMan Excel file not found: {TRACKMAN_EXCEL}. Update TRACKMAN_EXCEL in the first cell with the full path to your workbook.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("TrackMan Excel:", TRACKMAN_EXCEL.resolve())
print("Report output:", OUTPUT_DIR.resolve())

TrackMan Excel: C:\Users\brend\OneDrive\School OneDrive\2025 CCBL Trackman Data.xlsx
Report output: C:\Users\brend\OneDrive\School OneDrive\Jupyter\Falmouth_Pitcher_Reports


In [2]:
from lxml import etree
from zipfile import ZipFile
from pathlib import Path
from collections import Counter
import csv, re, json, argparse, time

S='{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
DEFAULT_PITCHERS=['Reich, Ryan','Harrington, Cooper','Shurtleff, Thomas','Arther, Adam','Doran, Scott']

def extract(source:Path,output:Path,pitchers:list[str]):
    source=Path(source).expanduser(); output=Path(output).expanduser()
    if not source.exists():raise FileNotFoundError(source)
    selected={name.strip().casefold() for name in pitchers}
    output.parent.mkdir(parents=True,exist_ok=True)
    with ZipFile(source) as z:
        shared=[]
        with z.open('xl/sharedStrings.xml') as f:
            for _,elem in etree.iterparse(f,events=('end',),tag=S+'si'):
                shared.append(''.join(t.text or '' for t in elem.iter(S+'t')))
                elem.clear()
        def value(c):
            e=c.find(S+'v')
            if e is None:
                v=c.find(S+'is')
                return ''.join(t.text or '' for t in v.iter(S+'t')) if v is not None else ''
            return shared[int(e.text)] if c.get('t')=='s' else (e.text or '')
        headers=[]; positions={}; counts=Counter(); scanned=0
        with z.open('xl/worksheets/sheet1.xml') as f,output.open('w',newline='',encoding='utf-8') as g:
            writer=csv.writer(g)
            for _,row in etree.iterparse(f,events=('end',),tag=S+'row'):
                rowno=int(row.get('r'))
                if rowno==1:
                    for cell in row.iter(S+'c'):
                        letters=re.match(r'[A-Z]+',cell.get('r')).group()
                        ix=0
                        for char in letters:ix=ix*26+ord(char)-64
                        positions[ix]=value(cell)
                    headers=sorted(positions)
                    writer.writerow(['WorkbookRow']+[positions[i] for i in headers])
                else:
                    scanned+=1
                    pitcher=''
                    for cell in row.iter(S+'c'):
                        ref=cell.get('r')
                        letters=re.match(r'[A-Z]+',ref).group()
                        if letters=='F':pitcher=value(cell);break
                        if len(letters)>1 or letters>'F':break
                    if pitcher.casefold() in selected:
                        cells={}
                        for cell in row.iter(S+'c'):
                            letters=re.match(r'[A-Z]+',cell.get('r')).group()
                            ix=0
                            for char in letters:ix=ix*26+ord(char)-64
                            if ix in positions:cells[ix]=value(cell)
                        writer.writerow([rowno]+[cells.get(i,'') for i in headers])
                        counts[pitcher]+=1
                row.clear()
                while row.getprevious() is not None:del row.getparent()[0]
    info={'source_excel_file':source.name,'source_sheet':'sheet1',
          'scanned_rows':scanned,'extracted_rows':sum(counts.values()),
          'pitcher_rows':dict(counts),'selected_pitchers':pitchers}
    output.with_name('trackman_extraction_audit.json').write_text(json.dumps(info,indent=2))
    return info

In [3]:
ryan_reich = dict(
    name='Ryan Reich',
    number='6',
    throws='RHP',
    school='Seton Hall',
    stats=dict(
        G='5', GS='2', IP='8.2', ERA='5.19', WHIP='1.73', SO='9', BB='5', HBP='0', WP='0',
    ),
    notes=[
        (
            'FB / Four-Seam',
            '87–90 Cut fastball with great ride. Attacks the zone. Throws a lot when behind. Gets '
            'whiffs, pitches up for them. Main strikeout pitch. 74% usage.'
        ),
        (
            'SL / Slider',
            '78–81 Bullet slider with some sweep. Hangs over the plate to LHH. Gets weak contact. '
            'Attacks the zone. 15% usage.'
        ),
        (
            'SP / Splitter',
            '77–79 Low-spin splitter, movement varies. Gets whiffs. Struggles with location. Has '
            'only thrown when ahead. 10% usage.'
        ),
    ],
    heatmaps_per_hand=dict(RHH=2, LHH=2),
)


cooper_harrington = dict(
    name='Cooper Harrington',
    number='10',
    throws='RHP',
    school='Liberty',
    stats=dict(
        G='7', GS='0', IP='9.2', ERA='0.93', WHIP='0.83', SO='11', BB='1', HBP='1', WP='1',
    ),
    notes=[
        (
            'FB / Four-Seam',
            '89–91 Two-seam fastball with decent ride and good ASR. Attacks the zone. Gets whiffs '
            'up in the zone. Works outside to LHH. Low release height (5.2) with great extension '
            '(6.8). 65% usage.'
        ),
        (
            'CT / Cutter',
            '85–87 Traditional cutter with a little less cut. Works away to RHH. Gets whiffs. '
            'Mostly throws when ahead. 7.5% usage.'
        ),
        (
            'SL / Slider',
            '77–79 Plays like a sweeper. Works down and away to RHH. 17% usage.'
        ),
        (
            'CH / Changeup',
            '78–81 Plays well off the fastball, drops off the table, similar ASR. Only thrown to '
            'LHH, works down and away. Gets weak contact and whiffs. 10% usage.'
        ),
    ],
    heatmaps_per_hand=dict(RHH=4, LHH=3),
)


thomas_shurtleff = dict(
    name='Thomas Shurtleff',
    number='15',
    throws='RHP',
    school='Penn',
    stats=dict(
        G='6', GS='0', IP='13.0', ERA='3.46', WHIP='1.08', SO='11', BB='4', HBP='2', WP='1',
    ),
    notes=[
        (
            'SNK / Sinker',
            '89–92, good run with varying amounts of sink with average extension out of long 3/4 '
            'slot. 31% usage to RHHs and 39% to LHHs. Will locate throughout plate but likes to '
            'target middle and outer third.'
        ),
        (
            'CT / Cutter',
            '84–87, bulletish shape with good sweep playing off sinker. 27% usage to all hitters. '
            'Will locate middle and outer third to RHHs and throughout plate to LHHs.'
        ),
        (
            'SL / Slider',
            '79–83, sweeper shape with not much drop (positive IVB). Main pitch to RHHs at 38% '
            'usage, will locate down and away. Does not throw to LHHs. Best chase pitch.'
        ),
        (
            'FB / Four-Seam',
            '89–92, flat with some run. Barely throws to RHHs, but 29% usage vs. LHHs. Likes to '
            'work away.'
        ),
        (
            'CH / Changeup',
            '82–84, decent sink and good run. Barely throws (lack of command), but something to '
            'keep in back of mind.'
        ),
    ],
    heatmaps_per_hand=dict(RHH=3, LHH=3),
)


adam_arther = dict(
    name='Adam Arther',
    number='16',
    throws='LHP',
    school='Kansas State',
    stats=dict(
        G='6', GS='0', IP='12.1', ERA='2.19', WHIP='1.54', SO='12', BB='5', HBP='1', WP='0',
    ),
    notes=[
        (
            'SNK / Sinker',
            '83–87, avg 84.2 mph. Good ASR from a quick, side-arm slot. Works middle/in (arm side) '
            'vs LHH and misses down and away vs LHH. Works arm-side/outer third vs RHH. Can throw '
            'it for a strike; definitely works bottom half of the zone.'
        ),
        (
            'SL / Slider',
            '76–78 mph. Expect SNK/SL vs LHH with occasional CH. Mainly works low and away vs LHH '
            'and tries to generate a chase. Has thrown it for a strike in on LHH too. Does not '
            'really throw the SL all that much vs RHH; again works low and away.'
        ),
        (
            'CH / Changeup',
            'Avg 77 mph. Works outer third vs LHH. Does not throw it all that often. SNK/CH heavy '
            'vs RHH; really goes for that arm-side movement profile on the bottom/outer third.'
        ),
    ],
    heatmaps_per_hand=dict(RHH=2, LHH=2),
)


scott_doran = dict(
    name='Scott Doran',
    number='21',
    throws='RHP',
    school='Coastal',
    stats=dict(
        G='3', GS='0', IP='6.1', ERA='4.26', WHIP='1.58', SO='5', BB='7', HBP='1', WP='3',
    ),
    notes=[
        (
            'Cut-FB / Fastball',
            'Cut fastball, 88–91, sharp cut with more drop compared to ride heater out of high 3/4 '
            'slot. High release point. Throws 41% to all hitters. Tends to locate up.'
        ),
        (
            'Ride-FB / Fastball',
            '87–90, good ride with some late cut. 22% usage vs. all hitters. Locates middle and '
            'upper thirds.'
        ),
        (
            'CB / Curveball',
            '77–81, 12–6 shape. Main secondary with 33% usage, can leave in zone.'
        ),
        (
            'CH / Changeup',
            '82–84, low spin with okay sink and run. Throws rarely, only to LHHs, but something to '
            'keep in back of mind.'
        ),
    ],
    heatmaps_per_hand=dict(RHH=2, LHH=2),
)


HISTORICAL_DATA = [
    ryan_reich,
    cooper_harrington,
    thomas_shurtleff,
    adam_arther,
    scott_doran,
]

In [4]:
import io, json, re, csv, math, statistics, collections
from pathlib import Path
import fitz
from PIL import Image
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader, simpleSplit
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from pptx import Presentation
from pptx.util import Inches

HERE = Path.cwd()
ROOT = OUTPUT_DIR.resolve()
SOURCE_CSV = ROOT/'source'/'harwich_five_raw_trackman.csv'
PDF_DIR = ROOT/'reports'/'pdf'; PPTX_DIR = ROOT/'reports'/'pptx'; PREV_DIR = ROOT/'reports'/'previews'
for _d in [PDF_DIR, PPTX_DIR, PREV_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

FONT_REGULAR, FONT_BOLD = 'Helvetica', 'Helvetica-Bold'
FONT_PAIRS = [
    ('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf','/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'),
    ('C:/Windows/Fonts/arial.ttf', 'C:/Windows/Fonts/arialbd.ttf'),
    ('/Library/Fonts/Arial.ttf', '/Library/Fonts/Arial Bold.ttf'),
]
for reg, bold in FONT_PAIRS:
    if Path(reg).exists() and Path(bold).exists():
        pdfmetrics.registerFont(TTFont('DV', reg))
        pdfmetrics.registerFont(TTFont('DVB', bold))
        FONT_REGULAR, FONT_BOLD = 'DV', 'DVB'
        break

W, H = 14*72, 8*72
MAROON='#842638'; PAPER='#F4F6F9'; WHITE='#FFFFFF'; INK='#111111'; BODY='#1C2833'; MUTED='#5D6875'; LINE='#D6DDE6'; LIGHT='#EDF1F5'; BLACK='#000000'
HAND_BAR='#0F0F10'
PITCH_COLORS={'Four-Seam':'#D94D50','Sinker':'#E3941B','Slider':'#9270C5','Cutter':'#A94C53','Changeup':'#0A9A88','Curveball':'#358DD1','Splitter':'#158A93'}

def xy(c, x, y):
    return x*72, H-y*72

def rect(c, x, y, w, h, fill=WHITE, stroke=None, r=0.0, sw=1):
    c.setFillColor(fill)
    c.setStrokeColor(stroke or fill)
    c.setLineWidth(sw)
    if r:
        c.roundRect(x*72, H-(y+h)*72, w*72, h*72, r*72, fill=1, stroke=bool(stroke))
    else:
        c.rect(x*72, H-(y+h)*72, w*72, h*72, fill=1, stroke=bool(stroke))

def t(c, x, y, content, size=9, col=BODY, bold=False, align='left'):
    c.setFont(FONT_BOLD if bold else FONT_REGULAR, size)
    c.setFillColor(col)
    fx, fy = xy(c, x, y); fy -= size
    {'left': c.drawString, 'right': c.drawRightString, 'center': c.drawCentredString}[align](fx, fy, str(content))

def wrap(c, text, x, y, max_w, fontsize=8.4, leading=12.2, col=BODY, bold=False, max_h=None):
    lines = simpleSplit(str(text), FONT_BOLD if bold else FONT_REGULAR, fontsize, max_w*72)
    drawn = 0
    for n, line in enumerate(lines):
        if max_h and (n+1)*leading > max_h*72:
            break
        t(c, x, y + n*leading/72, line, fontsize, col, bold)
        drawn += 1
    return drawn*leading/72

def panel(c, x, y, w, h, title=None, minor=None):
    rect(c, x, y, w, h, WHITE, LINE)
    rect(c, x, y, 0.042, h, BLACK)
    if title:
        t(c, x+.18, y+.14, title, 9.55, INK, True)
    if minor:
        t(c, x+w-.18, y+.15, minor, 7.5, MUTED, False, 'right')

                                                                        
                                                                               
PITCH_MAP = {'fastball':'Four-Seam','fourseam':'Four-Seam','four-seam':'Four-Seam',
             'fourseamfastball':'Four-Seam','two-seam':'Sinker',
             'twoseamfastball':'Sinker','sinker':'Sinker','slider':'Slider',
             'changeup':'Changeup','curveball':'Curveball','cutter':'Cutter',
             'splitter':'Splitter','sweeper':'Slider'}

def as_float(x):
    try:
        v=float(x)
        return v if math.isfinite(v) else None
    except (ValueError,TypeError):
        return None

def load_trackman():
    result=collections.defaultdict(list)
    if not SOURCE_CSV.is_file(): raise FileNotFoundError(SOURCE_CSV)
    with SOURCE_CSV.open(newline='',encoding='utf-8') as f:
        for raw in csv.DictReader(f):
            name=(raw.get('Pitcher') or '').strip()
            if ',' in name:
                last,first=(x.strip() for x in name.split(',',1))
                name=first+' '+last
            pitch=PITCH_MAP.get((raw.get('TaggedPitchType') or '').lower().strip())
            if not pitch: continue
            hand=(raw.get('BatterSide') or '').strip()
            if hand not in ('Right','Left'): continue
            record={'pitch':pitch,'hand':hand,'throw':raw.get('PitcherThrows'),
                    'WorkbookRow':raw.get('WorkbookRow'),
                    'RelHeight':as_float(raw.get('RelHeight')),
                    'RelSide':as_float(raw.get('RelSide')),
                    'HorzBreak':as_float(raw.get('HorzBreak')),
                    'InducedVertBreak':as_float(raw.get('InducedVertBreak')),
                    'RelSpeed':as_float(raw.get('RelSpeed')),
                    'SpinRate':as_float(raw.get('SpinRate')),
                    'Extension':as_float(raw.get('Extension')),
                    'PlateLocHeight':as_float(raw.get('PlateLocHeight')),
                    'PlateLocSide':as_float(raw.get('PlateLocSide')),
                    'PitchCall':(raw.get('PitchCall') or '').strip(),
                    'Date':(raw.get('Date') or '').strip()}
            result[name].append(record)
    return result

PITCH_DATA={}
def rows_for(rec): return PITCH_DATA[rec['name']]

def pitch_arsenal(rows):
    groups=collections.defaultdict(list)
    for r in rows: groups[r['pitch']].append(r)
    def ave(rows,key):
        values=[r[key] for r in rows if r[key] is not None]
        return statistics.mean(values) if values else float('nan')
    return [[pitch,len(rs),ave(rs,'RelSpeed'),ave(rs,'InducedVertBreak'),
             ave(rs,'HorzBreak'),ave(rs,'SpinRate'),ave(rs,'Extension')]
            for pitch,rs in sorted(groups.items(),key=lambda kv:-len(kv[1]))]

def usage_pct(rows,hand,pitch):
    s=[r for r in rows if r['hand']==hand]
    return (100.0*sum(r['pitch']==pitch for r in s)/len(s)) if s else None

def fig_to_pil(fig):
    buf=io.BytesIO()
    fig.savefig(buf,format='png',dpi=190,facecolor='white',bbox_inches=None,pad_inches=0)
    plt.close(fig); buf.seek(0)
    return Image.open(buf).convert('RGB')

def plot_release(rows):
    fig,ax=plt.subplots(figsize=(5.1,3.55),dpi=190)
    fig.subplots_adjust(left=.10,right=.985,bottom=.115,top=.91)
    for pitch in dict.fromkeys(r['pitch'] for r in rows):
        points=[r for r in rows if r['pitch']==pitch and r['RelSide'] is not None and r['RelHeight'] is not None]
                                                                             
                                                                             
                                                            
        ax.scatter([-r['RelSide'] for r in points],[r['RelHeight'] for r in points],
                   s=16,alpha=.84,color=PITCH_COLORS.get(pitch,'#677588'),linewidths=0,zorder=5)
    ax.add_patch(Polygon([(-.72,.79),(.72,.79),(.72,.56),(0,.23),(-.72,.56)],
                         facecolor='#D9E2E9',edgecolor='#67788C',lw=.8,zorder=3))
    ax.axvline(0,ls='--',lw=.7,color='#A7B4C4')
    ax.set_xlim(-5.12,5.12); ax.set_ylim(0,8.0)
    ax.set_xticks(np.arange(-5,6,1)); ax.set_yticks(np.arange(0,9,1))
    ax.grid(color='#E9EEF3',lw=.65)
    ax.tick_params(labelsize=8,colors='#57697F',length=0)
    for sp in ax.spines.values():sp.set_visible(False)
    ax.set_aspect('equal',adjustable='box')
    return fig_to_pil(fig)

def plot_movement(rows):
    fig,ax=plt.subplots(figsize=(5.3,3.55),dpi=190)
    fig.subplots_adjust(left=.12,right=.96,bottom=.13,top=.94)
    valid=[r for r in rows if r['HorzBreak'] is not None and r['InducedVertBreak'] is not None]
    for pitch in dict.fromkeys(r['pitch'] for r in rows):
        pts=[r for r in valid if r['pitch']==pitch]
        ax.scatter([r['HorzBreak'] for r in pts],[r['InducedVertBreak'] for r in pts],
                   s=14,alpha=.74,color=PITCH_COLORS.get(pitch,'#677588'),linewidths=0,zorder=4)
    if valid:
        x=np.array([r['HorzBreak'] for r in valid]); y=np.array([r['InducedVertBreak'] for r in valid]);
        xmax=max(25,math.ceil(max(abs(np.quantile(x,.005)),abs(np.quantile(x,.995)))+3))
        ymax=max(25,math.ceil(max(abs(np.quantile(y,.005)),abs(np.quantile(y,.995)))+3))
    else:xmax=30; ymax=30
    ax.set_xlim(-xmax,xmax); ax.set_ylim(-ymax,ymax)
    ax.axhline(0,lw=.85,color='#9EACBD');ax.axvline(0,lw=.85,color='#9EACBD')
    ax.grid(color='#E9EEF3',lw=.7)
    ax.tick_params(labelsize=8,colors='#57697F',length=0)
    for sp in ax.spines.values():sp.set_visible(False)
    ax.set_aspect('equal',adjustable='box')
    return fig_to_pil(fig)

HEAT_CMAP=LinearSegmentedColormap.from_list('FalmouthDensity',
                ['#1A598B','#619CC4','#DAE6EF','#FFF9F8','#DF9DA1','#BA5264','#842638'])
def plot_location(rows,pitch,hand):
    sub=[r for r in rows if r['pitch']==pitch and r['hand']==hand and
         r['PlateLocSide'] is not None and r['PlateLocHeight'] is not None]
    fig,ax=plt.subplots(figsize=(3.05,3.65),dpi=190)
    fig.subplots_adjust(left=.035,right=.965,bottom=.03,top=.97)
    ax.set_xlim(-2.20,2.20);ax.set_ylim(.0,5.0)
    ax.set_facecolor('#F3F6F9')
    xx=np.array([-r['PlateLocSide'] for r in sub],dtype=float)
    yy=np.array([r['PlateLocHeight'] for r in sub],dtype=float)
    if len(sub)>=12:
        counts,ex,ey=np.histogram2d(xx,yy,bins=(76,90),range=[[-2.2,2.2],[0,5]])
        smooth=gaussian_filter(counts,sigma=(5.0,6.0))
        vmax=max(float(smooth.max()),1e-10)
        density=np.clip(smooth/vmax,0,1)
                                                                                 
        support_alpha=np.clip((density-.003)/.012,0,1)
        ax.imshow(np.power(density,.70).T,origin='lower',extent=[-2.2,2.2,0,5],
                  cmap=HEAT_CMAP,vmin=.0,vmax=1.0,aspect='auto',interpolation='bilinear',
                  alpha=support_alpha.T)
    else:
        ax.scatter(xx,yy,s=18,c=MAROON,alpha=.78,edgecolors='white',linewidths=.35)
    ax.add_patch(Rectangle((-.83,1.5),1.66,2,fill=False,ec=INK,lw=1.25,zorder=8))
    ax.set_aspect('equal',adjustable='box')
    ax.set_xticks([]);ax.set_yticks([])
    for sp in ax.spines.values():sp.set_visible(False)
    return fig_to_pil(fig),len(sub)

def contain(c, img, x, y, w, h):
    iw, ih = img.size
    scale = min(w/iw, h/ih)
    dw, dh = iw*scale, ih*scale
    px = x + (w-dw)/2; py = y + (h-dh)/2
    c.drawImage(ImageReader(img), px*72, H-(py+dh)*72, dw*72, dh*72, mask='auto')

def topbar(c, rec, pageno):
                                                                              
    rect(c, 0, 0, 14, 1.04, MAROON, MAROON)
    t(c, .48, .426, 'FALMOUTH COMMODORES', 12.5, BLACK, True)
    name_size = 21.0 if len(rec['name']) <= 22 else 19.3
    t(c, 7.02, .17, rec['name'], name_size, WHITE, True, 'center')
    t(c, 7.02, .69, f"#{rec['number']}  •  {rec['throws']}  •  {rec['school']}",
      9.2, '#F3E6EB', False, 'center')
    t(c, 13.53, .29, 'PITCHER ADVANCE SCOUTING', 10.0, WHITE, True, 'right')
    t(c, 13.53, .70, f'PAGE {pageno} OF 2', 8.0, '#F3E6EB', False, 'right')
    rect(c, 0, 1.04, 14, 6.96, PAPER)

def footer(c, pageno):
    rect(c, .47, 7.69, 13.06, .012, LINE)
    t(c,.48,7.76,'TRACKMAN EXCEL: PLOTS / ARSENAL / USAGE  •  HISTORICAL PDF: GAME STATS / SCOUTING NOTES',6.6,MUTED)
    t(c, 13.53, 7.76, f'PAGE {pageno} / 2', 7, MUTED, False, 'right')

def stats(c, rec):
    labels=['G','GS','IP','ERA','WHIP','SO','BB','HBP','WP']
    start, gap = .46, .10
    width = (13.08-8*gap)/9
    for j, key in enumerate(labels):
        x = start+j*(width+gap)
        rect(c, x, 1.21, width, .65, WHITE, LINE)
        midpoint=x+width/2
        t(c, midpoint, 1.32, key, 8.0, MUTED, True, 'center')
        t(c, midpoint, 1.53, rec['stats'][key], 15.2, BODY, True, 'center')

def draw_release_legend(c, rec, x, y, width=1.05):
    t(c,x,y,'PITCH TYPES',6.9,MUTED,True)
    for j,(pitch,*_) in enumerate(rec['arsenal'][:7]):
        yy=y+.23+j*.255
        c.setFillColor(PITCH_COLORS.get(pitch,BODY))
        c.circle((x+.075)*72,H-(yy+.052)*72,3.0,stroke=0,fill=1)
        t(c,x+.19,yy,pitch,7.45,BODY)

def cropped_plots(c,rec,page=None):
    panel(c,.46,1.99,5.92,3.04,'RELEASE POINTS',"Hitter's view")
    panel(c,7.42,1.99,6.12,3.04,'PITCH MOVEMENT','HB / IVB')
    rows=rows_for(rec)
    contain(c,plot_release(rows),.64,2.32,4.65,2.58)
    draw_release_legend(c,rec,6.58,2.62)
    contain(c,plot_movement(rows),8.22,2.30,4.28,2.61)

def pitch_table(c,rec):
    panel(c,.46,5.17,8.45,2.31,'PITCH ARSENAL')
                                                                        
    bounds=[.64,2.78,3.69,4.51,5.32,6.17,7.37,8.73]
    centers=[(bounds[i]+bounds[i+1])/2 for i in range(len(bounds)-1)]
    headers=['PITCH','N','VELO','IVB','HB','SPIN','EXT']
    rect(c,.64,5.56,8.09,.31,LIGHT)
    for key,x in zip(headers,centers):
        t(c,x,5.63,key,8.0,MUTED,True,'center')
    for i,ar in enumerate(rec['arsenal'][:7]):
        pitch,ct,velo,ivb,hb,spin,ext=ar
        yy=5.94+i*(.223 if len(rec['arsenal']) >= 7 else .25)
        if i%2:rect(c,.64,yy-.014,8.09,(.223 if len(rec['arsenal']) >= 7 else .25),'#F7F9FB')
        row=[pitch,str(ct),f'{velo:.1f}',f'{ivb:.1f}',f'{hb:.1f}',f'{spin:.0f}',f'{ext:.1f}']
        for j,x in enumerate(centers):
            t(c,x,yy,row[j],8.65,PITCH_COLORS.get(pitch,BODY) if j==0 else BODY,j==0,'center')

def usage_table(c,rec):
    panel(c,9.05,5.17,4.48,2.31,'USAGE BY BATTER HAND')
    rect(c,9.23,5.56,4.12,.31,LIGHT)
    ubounds=[9.23,11.12,12.31,13.35]
    ucenters=[(ubounds[i]+ubounds[i+1])/2 for i in range(3)]
    for label,x in zip(('PITCH','RHH','LHH'),ucenters):
        t(c,x,5.63,label,7.75,MUTED,True,'center')
    for i,(pitch,*_) in enumerate(rec['arsenal'][:7]):
        yy=5.94+i*(.223 if len(rec['arsenal']) >= 7 else .25)
        if i%2:rect(c,9.23,yy-.014,4.12,(.223 if len(rec['arsenal']) >= 7 else .25),'#F7F9FB')
        t(c,ucenters[0],yy,pitch,8.65,PITCH_COLORS.get(pitch,BODY),True,'center')
        for hand,x in [('RHH',ucenters[1]),('LHH',ucenters[2])]:
            u=rec['usage'][hand].get(pitch)
            t(c,x,yy,'—' if u is None else f'{u:.1f}%',8.55,BODY,False,'center')

def page_one(c, rec, page):
    topbar(c, rec, 1)
    stats(c, rec)
    cropped_plots(c, rec, page)
    pitch_table(c, rec)
    usage_table(c, rec)
    footer(c, 1)
    c.showPage()

def map_card(c,pitch,coords,page,x,y,w,h,usage=None,rows=None,hand=None):
    rect(c,x,y,w,h,WHITE,LINE)
    rect(c,x,y,.035,h,BLACK)
    t(c,x+.14,y+.12,pitch.upper(),8.15,PITCH_COLORS.get(pitch,BODY),True)
    im,n=plot_location(rows,pitch,hand)
    contain(c,im,x+.07,y+.30,w-.14,h-.52)
    t(c,x+w/2,y+h-.16,(f'{n} pitches  •  ' + ('—' if usage is None else f'{usage:.1f}% use')),
      7.2,MUTED,False,'center')

def map_strip(c,rec,page,hand,x,y,w):
    label='VS RIGHT HANDED HITTERS' if hand=='RHH' else 'VS LEFT HANDED HITTERS'
    rect(c,x,y,w,.31,HAND_BAR,HAND_BAR)
    t(c,x+.16,y+.085,label,9.1,WHITE,True)
    rows=rows_for(rec)
    hand_full={'RHH':'Right','LHH':'Left'}[hand]
    rank=collections.Counter(r['pitch'] for r in rows if r['hand']==hand_full)
    n_cards=rec['heatmaps_per_hand'][hand]                                            
    m=[p for p,count in rank.most_common(n_cards)]
    g=.16;n=len(m)
    maxwidth={2:3.23,3:2.64,4:2.02}.get(n,2.02)
    cardw=min(maxwidth,(w-(n-1)*g)/n)
    total=n*cardw+(n-1)*g
    left=x+(w-total)/2
    for i,pitch in enumerate(m):
        use=usage_pct(rows,hand_full,pitch)
        map_card(c,pitch,None,None,left+i*(cardw+g),y+.37,cardw,2.64,use,rows,hand_full)

def notes_card(c, rec):

    x, y, width, height = 9.18, 1.36, 4.35, 6.00
    panel(c, x, y, width, height, 'PITCH BY PITCH SCOUTING NOTES')
    text_x = x + .20
    line_width = width - .40
    top = y + .55
    bottom = y + height - .19

    body_size = 9.55
    label_size = 9.05
    leading = 14.0
    label_to_body = .245                                         
    section_gap = {3: .47, 4: .33, 5: .22}.get(len(rec['notes']), .25)

    def layout_for(font_size, between):
                                                                              
        entries=[]
        for label, body in rec['notes']:
            lines = simpleSplit(str(body), FONT_REGULAR, font_size, line_width*72)
            entries.append((label, lines))
        used = sum(label_to_body + len(lines)*(leading/72) for _,lines in entries)
        used += between*max(0,len(entries)-1)
        return entries, used

    entries, required = layout_for(body_size, section_gap)
    usable = bottom - top
    if required > usable:
                                                                            
        section_gap = .17
        entries, required = layout_for(body_size, section_gap)
    if required > usable:
        raise ValueError(f'Scouting notes overflow for {rec["name"]}: {required:.2f} > {usable:.2f} in')

    yy = top
    for i, (label, lines) in enumerate(entries):
        t(c, text_x, yy, label.upper(), label_size, MAROON, True)
        body_y = yy + label_to_body
        for j, line in enumerate(lines):
            t(c, text_x, body_y + j*leading/72, line, body_size, BODY)
        yy = body_y + len(lines)*leading/72
        if i != len(entries)-1:
            yy += section_gap
                                                                
    assert yy <= bottom + 0.001, (rec['name'], yy, bottom)

def page_two(c, rec, page):
    topbar(c, rec, 2)
    map_strip(c, rec, page, 'RHH', .46, 1.33, 8.54)
    map_strip(c, rec, page, 'LHH', .46, 4.46, 8.54)
    notes_card(c, rec)
    footer(c, 2)
    c.showPage()

def make_pptx(pdf_file, pptx_file, preview_stem):
    pdf = fitz.open(pdf_file); assert len(pdf) == 2
    pres = Presentation(); pres.slide_width = Inches(14); pres.slide_height = Inches(8)
    for i, p in enumerate(pdf):
        img = p.get_pixmap(matrix=fitz.Matrix(1.8,1.8), alpha=False)
        path = PREV_DIR/(preview_stem+f'_page{i+1}.png'); img.save(path)
        slide = pres.slides.add_slide(pres.slide_layouts[6])
        slide.shapes.add_picture(str(path), 0, 0, width=pres.slide_width, height=pres.slide_height)
    pres.save(pptx_file)
    return len(pres.slides)

def build_combined_pdf(pdf_dir: Path, out_file: Path):
    combo = fitz.open()
    for pdf_path in sorted(pdf_dir.glob('*_Pitcher_Advance_2Page.pdf')):
        src = fitz.open(pdf_path)
        combo.insert_pdf(src)
        src.close()
    combo.save(out_file)
    combo.close()

def main():
    global PITCH_DATA
    PITCH_DATA=load_trackman()
    data = copy.deepcopy(HISTORICAL_DATA)
    assert len(data) == 5
    for rec in data:
        if rec['name'] not in PITCH_DATA: raise KeyError(f'Pitcher not found in workbook extraction: {rec["name"]}')
        rec['arsenal']=pitch_arsenal(PITCH_DATA[rec['name']])
        rec['usage']={h:{p[0]:usage_pct(PITCH_DATA[rec['name']],hand,p[0]) for p in rec['arsenal']} for h,hand in [('RHH','Right'),('LHH','Left')]}
        stem = re.sub(r'[^A-Za-z0-9]+','_',rec['name']).strip('_')
        pdf_file = PDF_DIR/(stem+'_Pitcher_Advance_2Page.pdf')
        pptx_file = PPTX_DIR/(stem+'_Pitcher_Advance_2Page.pptx')
        c = canvas.Canvas(str(pdf_file), pagesize=(W,H), pageCompression=1)
        c.setTitle(rec['name']+' | Falmouth pitcher advance scouting')
        c.setAuthor('Falmouth Commodores / Baseball Operations')
        page_one(c, rec, None); page_two(c, rec, None); c.save()
        assert len(fitz.open(pdf_file)) == 2
        assert make_pptx(pdf_file, pptx_file, stem) == 2
        print('Created:', rec['name'], 'TrackMan pitches:',len(PITCH_DATA[rec['name']]),'pitch types:',len(rec['arsenal']),flush=True)
    build_combined_pdf(PDF_DIR, PDF_DIR/'All_Five_Harwich_Pitchers_Combined.pdf')
    print('COMPLETE: five pitchers with Excel-derived charts')

c:\Users\brend\AppData\Local\Programs\Python\Python313\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [5]:
EXTRACTED_CSV = OUTPUT_DIR / "source" / "harwich_five_raw_trackman.csv"
AUDIT = extract(TRACKMAN_EXCEL, EXTRACTED_CSV, PITCHERS)
print(json.dumps(AUDIT, indent=2))

{
  "source_excel_file": "2025 CCBL Trackman Data.xlsx",
  "source_sheet": "sheet1",
  "scanned_rows": 114410,
  "extracted_rows": 1573,
  "pitcher_rows": {
    "Arther, Adam": 435,
    "Doran, Scott": 317,
    "Harrington, Cooper": 156,
    "Reich, Ryan": 370,
    "Shurtleff, Thomas": 295
  },
  "selected_pitchers": [
    "Reich, Ryan",
    "Harrington, Cooper",
    "Shurtleff, Thomas",
    "Arther, Adam",
    "Doran, Scott"
  ]
}


In [6]:
main()
print("Individual PDF reports:", PDF_DIR.resolve())
print("Individual PowerPoint reports:", PPTX_DIR.resolve())
print("Combined five-pitcher PDF:", (PDF_DIR / "All_Five_Harwich_Pitchers_Combined.pdf").resolve())

Created: Ryan Reich TrackMan pitches: 365 pitch types: 6
Created: Cooper Harrington TrackMan pitches: 151 pitch types: 5
Created: Thomas Shurtleff TrackMan pitches: 295 pitch types: 5
Created: Adam Arther TrackMan pitches: 435 pitch types: 5
Created: Scott Doran TrackMan pitches: 312 pitch types: 7
COMPLETE: five pitchers with Excel-derived charts
Individual PDF reports: C:\Users\brend\OneDrive\School OneDrive\Jupyter\Falmouth_Pitcher_Reports\reports\pdf
Individual PowerPoint reports: C:\Users\brend\OneDrive\School OneDrive\Jupyter\Falmouth_Pitcher_Reports\reports\pptx
Combined five-pitcher PDF: C:\Users\brend\OneDrive\School OneDrive\Jupyter\Falmouth_Pitcher_Reports\reports\pdf\All_Five_Harwich_Pitchers_Combined.pdf
